# Problema de clasificación Supervisada


### Descripción del problema:

Predicción del salario de una persona en función a sus características.

El objetivo del problema es predecir si una persona tiene un salario de más de 50 mil dólares anuales o no, en base a sus características.

Haremos uso del dataset Adult. Este dataset proviene de la siguiente ruta de la University of California Irvine (**Url:** https://archive.ics.uci.edu/ml/datasets/Census+Income)

### Descripción del dataset:

Cuenta con un total de **14 variables predictoras X** y una **variable continua a predecir Y**.

El número total de muestras es de 32561 personas.

**Información de las variables:**

**Variable dependiente Y:**
TARGET: >50K, <=50K.

**Variables independientes X:**
* age: continuous.
* workclass: Private, Self-emp-not-inc, Self-emp-inc, Federal-gov, Local-gov, State-gov, Without-pay, Never-worked.
* fnlwgt: continuous.
* education: Bachelors, Some-college, 11th, HS-grad, Prof-school, Assoc-acdm, Assoc-voc, 9th, 7th-8th, 12th, Masters, 1st-4th, 10th, Doctorate, 5th-6th, Preschool.
* education-num: continuous.
* marital-status: Married-civ-spouse, Divorced, Never-married, Separated, Widowed, Married-spouse-absent, Married-AF-spouse.
* occupation: Tech-support, Craft-repair, Other-service, Sales, Exec-managerial, Prof-specialty, Handlers-cleaners, Machine-op-inspct, Adm-clerical, Farming-fishing, Transport-moving, Priv-house-serv, Protective-serv, Armed-Forces.
* relationship: Wife, Own-child, Husband, Not-in-family, Other-relative, Unmarried.
* race: White, Asian-Pac-Islander, Amer-Indian-Eskimo, Other, Black.
* sex: Female, Male.
* capital-gain: continuous.
* capital-loss: continuous.
* hours-per-week: continuous.
* native-country: United-States, Cambodia, England, Puerto-Rico, Canada, Germany, Outlying-US(Guam-USVI-etc), India, Japan, Greece, South, China, Cuba, Iran, Honduras, Philippines, Italy, Poland, Jamaica, Vietnam, Mexico, Portugal, Ireland, France, Dominican-Republic, Laos, Ecuador, Taiwan, Haiti, Columbia, Hungary, Guatemala, Nicaragua, Scotland, Thailand, Yugoslavia, El-Salvador, Trinadad&Tobago, Peru, Hong, Holand-Netherlands.

# Carga de librerías:

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
import sklearn.metrics as metrics
from IPython.core.display import display, HTML
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Lectura de datos:

In [ ]:
# Defino la lista de nombres de las variables
nombres_columnas = ['edad', 'clase_trabajo', 'fnlwgt', 'educacion',
                   'educacion_num', 'estado_civil', 'ocupacion',
                    'relaciones', 'raza', 'sexo', 'ganancia_capital',
                    'perdida_capital', 'horas_por_semana', 'pais_nacimiento', 'target']
# Leo el dataset, el cual tiene un separador especial... lo normal es encontrarse ficheros
# separados por ',' o por ';'
XY = pd.read_csv('adult.data', sep=', ', names=nombres_columnas, index_col=False)

In [ ]:
XY[:2]

,edad,clase_trabajo,fnlwgt,educacion,educacion_num,estado_civil,ocupacion,relaciones,raza,sexo,ganancia_capital,perdida_capital,horas_por_semana,pais_nacimiento,target
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K


In [ ]:
print(u'- El número de filas en el dataset es: {}'.format(XY.shape[0]))
print(u'- El número de columnas en el dataset es: {}'.format(XY.shape[1]))
print(u'- Los nombres de las variables son: {}'.format(list(XY.columns)))
XY[:2]

- El número de filas en el dataset es: 32561
- El número de columnas en el dataset es: 15
- Los nombres de las variables son: ['edad', 'clase_trabajo', 'fnlwgt', 'educacion', 'educacion_num', 'estado_civil', 'ocupacion', 'relaciones', 'raza', 'sexo', 'ganancia_capital', 'perdida_capital', 'horas_por_semana', 'pais_nacimiento', 'target']


,edad,clase_trabajo,fnlwgt,educacion,educacion_num,estado_civil,ocupacion,relaciones,raza,sexo,ganancia_capital,perdida_capital,horas_por_semana,pais_nacimiento,target
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K


# Preprocesamiento de datos

## Missings:

Represento el conteo de valores faltantes por variable. En caso de haber, una de las formas de rellenarlos es:

    df['nombre_columna'] = df['nombre_columna'].fillna(valor)

In [ ]:
XY.isnull().sum()

,0
edad,0
clase_trabajo,0
fnlwgt,0
educacion,0
educacion_num,0
estado_civil,0
ocupacion,0
relaciones,0
raza,0
sexo,0


## Categóricas a numéricas:

Es necesario convertir las variables categóricas a numéricas de cara a introducirlas en modelos:

In [ ]:
# Lista de variables categóricas
XY.select_dtypes(exclude=['number']).columns

Index(['clase_trabajo', 'educacion', 'estado_civil', 'ocupacion', 'relaciones',
       'raza', 'sexo', 'pais_nacimiento', 'target'],
      dtype='object')

Este objeto codifica las variables categóricas a números distintos

In [ ]:
le = LabelEncoder()

### clase_trabajo:

In [ ]:
XY['clase_trabajo'].value_counts()

,count
clase_trabajo,
Private,22696
Self-emp-not-inc,2541
Local-gov,2093
?,1836
State-gov,1298
Self-emp-inc,1116
Federal-gov,960
Without-pay,14
Never-worked,7


Las muestras que sean distintas de Private los codifico como 0s, los Private como 1s.

In [ ]:
XY.loc[XY['clase_trabajo'] != 'Private', 'clase_trabajo'] = 0
XY.loc[XY['clase_trabajo'] == 'Private', 'clase_trabajo'] = 1
XY['clase_trabajo'] = XY['clase_trabajo'].astype(int)

### educacion

In [ ]:
XY['educacion'].value_counts()

,count
educacion,
HS-grad,10501
Some-college,7291
Bachelors,5355
Masters,1723
Assoc-voc,1382
11th,1175
Assoc-acdm,1067
10th,933
7th-8th,646


Esta es otra forma de cambiar los valores de una variable. Se realiza con un mapeo de valores 1 a 1.

In [ ]:
XY["educacion"].head()

,educacion
0,Bachelors
1,Bachelors
2,HS-grad
3,11th
4,Bachelors


In [ ]:
dic = {'Doctorate':0, 'Masters':1, 'Bachelors': 2, 'Some-college':3, 'Assoc-voc':4,
       'Assoc-acdm': 5, 'HS-grad': 6, 'Prof-school': 7, 'Preschool': 8,
       '12th': 9, '11th': 10, '10th': 11, '9th': 12, '7th-8th': 13,
       '5th-6th': 14, '1st-4th':15}

XY["educacion"].replace(dic, inplace=True)

In [ ]:
XY["educacion"].head()

,educacion
0,2
1,2
2,6
3,10
4,2


### estado_civil

In [ ]:
XY['estado_civil'].value_counts()

,count
estado_civil,
Married-civ-spouse,14976
Never-married,10683
Divorced,4443
Separated,1025
Widowed,993
Married-spouse-absent,418
Married-AF-spouse,23


Codifico con label encoder.

In [ ]:
XY.estado_civil= le.fit_transform(XY.estado_civil.values)

### ocupacion

In [ ]:
XY['ocupacion'].value_counts()

,count
ocupacion,
Prof-specialty,4140
Craft-repair,4099
Exec-managerial,4066
Adm-clerical,3770
Sales,3650
Other-service,3295
Machine-op-inspct,2002
?,1843
Transport-moving,1597


In [ ]:
XY.ocupacion= le.fit_transform(XY.ocupacion.values)

### relaciones

In [ ]:
XY['relaciones'].value_counts()

,count
relaciones,
Husband,13193
Not-in-family,8305
Own-child,5068
Unmarried,3446
Wife,1568
Other-relative,981


In [ ]:
XY.relaciones= le.fit_transform(XY.relaciones.values)

### raza

In [ ]:
XY['raza'].value_counts()

,count
raza,
White,27816
Black,3124
Asian-Pac-Islander,1039
Amer-Indian-Eskimo,311
Other,271


In [ ]:
XY.loc[XY.raza != 'White', 'raza'] = 0
XY.loc[XY.raza == 'White', 'raza'] = 1
XY.raza = XY.raza.astype(int)

### sexo

In [ ]:
XY['sexo'].value_counts()

,count
sexo,
Male,21790
Female,10771


In [ ]:
dic = {'Male':0, 'Female':1}
XY["sexo"].replace(dic, inplace=True)

### pais_nacimiento

In [ ]:
XY['pais_nacimiento'].value_counts()

,count
pais_nacimiento,
United-States,29170
Mexico,643
?,583
Philippines,198
Germany,137
Canada,121
Puerto-Rico,114
El-Salvador,106
India,100


In [ ]:
XY.loc[XY.pais_nacimiento != 'United-States', 'pais_nacimiento'] = 0
XY.loc[XY.pais_nacimiento == 'United-States', 'pais_nacimiento'] = 1
XY.pais_nacimiento = XY.pais_nacimiento.astype(int)

### target

In [ ]:
XY['target'].value_counts()

,count
target,
<=50K,24720
>50K,7841


La mayoría de las veces las targets hay que codificarlas.

En este caso, la target la codifico a 0s si es <50K y a 1s si es >50K.

In [ ]:
dic = {'<=50K':0, '>50K':1}
XY["target"].replace(dic, inplace=True)

## Comprobación tipos no numéricos:

Ya no tengo más tipos numéricos, continúo.

In [ ]:
XY.select_dtypes(exclude=['number']).columns

Index([], dtype='object')

## Division en features X + target Y

In [ ]:
X = XY.drop('target', axis=1)
Y = XY['target']

## Estandarización de los datos previa:

Como se comenta en la unidad, hay modelos que parten de la hipótesis que los datos son centrados y, por tanto, se necesita estandarizar. Suele ser una buena práctica porque no suele perjudicar.

Además, la inversa se puede realizar de forma sencilla.

In [ ]:
obj_escalar = StandardScaler()
X_estandarizado = obj_escalar.fit_transform(X)

## División en train y test:

El conjunto de test NUNCA se usa para ajustar los modelos. Es un conjunto que se separa y se valida al final del todo para obtener una métrica.

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(X_estandarizado, Y, test_size=0.2, random_state=0)

# Aplicamos un modelo de regresión logística

In [ ]:
modelo = LogisticRegression()
parametros = {"C": [0., 0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08,0.09],
              "class_weight":['balanced', None]}

Con GridSearchCV se realiza una optimización. Esta función lo que hace es ajustar el modelo que se pasa como argumento con todas las combinaciones posibles de los parámetros. En este caso, todas las combinaciones de **C** y **class_weights**.

In [ ]:
modelo_gs = GridSearchCV(modelo, param_grid=parametros,
                         cv = 5, scoring='roc_auc')
modelo_gs.fit(X_train, Y_train)

GridSearchCV(cv=5, estimator=LogisticRegression(),
             param_grid={'C': [0.0, 0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07,
                               0.08, 0.09],
                         'class_weight': ['balanced', None]},
             scoring='roc_auc')

In [ ]:
print(modelo_gs.best_params_, "\nROC AUC: {}".format(round(modelo_gs.best_score_,2)))

{'C': 0.09, 'class_weight': 'balanced'} 
ROC AUC: 0.86


In [ ]:
df_search = pd.DataFrame.from_dict(modelo_gs.cv_results_)

#### Analizando el modelo con el mejor alpha

En este paso nos quedamos con los mejores parámetros obtenidos en el paso anterior:

In [ ]:
reg_log =  LogisticRegression(C=modelo_gs.best_params_['C'],
                              class_weight=modelo_gs.best_params_['class_weight'])

Ajusto a todos los datos de entrenamiento.

In [ ]:
reg_log.fit(X_train, Y_train)

LogisticRegression(C=0.09, class_weight='balanced')

Aquí es cuando entra en juego el conjunto de Test. Cuando se quiere validar un modelo ya elegido y optimizado.

Con ese modelo optimizado, predigo test para ver cómo se comporta en datos que no ha visto antes.

In [ ]:
y_test_pred_prob = reg_log.predict_proba(X_test)
y_test_pred_prob_pos = y_test_pred_prob[np.where(Y_test == 1)[0]]
y_test_pred_prob_neg = y_test_pred_prob[np.where(Y_test == 0)[0]]

## Umbralizo las predicciones:

Las probabilidades que devuelve el modelo son valores continuos entre 0 y 1. Para pasarlo a 0s y a 1s es necesario usar un umbral de corte. Todo lo que sea mayor que el umbral será predicción = 1, y lo que sea menor será predicción = 0.

In [ ]:
umbral = 0.6
y_umbralizadas = 1*(y_test_pred_prob[:, 1] > umbral)

In [ ]:
print(u"Matriz de confusión\n", metrics.confusion_matrix(Y_test, y_umbralizadas))
print("\nAccuracy\t{}".format(round(metrics.accuracy_score(Y_test, y_umbralizadas),2)))
print("Sensitividad\t{}".format(round(metrics.recall_score(Y_test, y_umbralizadas),2)))
print(u"Precisión\t{}".format(round(metrics.precision_score(Y_test, y_umbralizadas),2)))

Matriz de confusión
 [[4132  786]
 [ 537 1058]]

Accuracy	0.8
Sensitividad	0.66
Precisión	0.57
